# sensitivity study for FDM-PINN

In [1]:
"""
FD step-size (h) sensitivity study for FDM-PINN, sinusoidal loading, SS-SS-SS-SS.

Addresses reviewer item: "FD step h=0.01 still needs validation. Your code
correctly notes that h=0.01=0.5*dx, but this has not yet been demonstrated
to be an appropriate choice. A small h-sensitivity study is needed before
using the FDM-PINN results as a strong paper claim."

This script trains a fresh FDM-PINN (same architecture/training pipeline as
fdm_pinn_sinusoidal.py) once per candidate h value, then reports:
  - final relative L2 displacement error vs h        -> accuracy trade-off
  - final center-deflection % error vs h              -> accuracy trade-off
  - mean training time per epoch vs h                  -> cost trade-off
  - a simple automatic recommendation for h

Expected shape of the accuracy-vs-h curve (classic FD truncation-vs-roundoff
trade-off): error should DECREASE as h shrinks (lower truncation error) down
to some point, then INCREASE again for very small h (float64 cancellation
in the numerator of the stencil, since w(x+h)-w(x) ~ h for tiny h). The
recommended h is the one at/near the minimum of this U-shaped curve.

Default h values tested: {0.125, 0.25, 0.5, 1.0, 2.0, 4.0} x grid spacing
Δx = 1/(nx-1) = 0.02, i.e. h in {0.0025, 0.005, 0.01, 0.02, 0.04, 0.08}.

NOTE ON RUNTIME: this trains N_H_VALUES full models. With MAX_EPOCHS=5000
(same budget as the main FDM-PINN run) this is N_H_VALUES x as expensive as
a single FDM-PINN run. On CPU this is slow -- run on the HPC GPU. For a
quick local sanity check, reduce MAX_EPOCHS/PATIENCE at the top of CONFIG.

Run:
    python fdm_h_sensitivity_study.py
"""

import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
LOADING_CASE = "sinusoidal"
OUT_DIR = os.path.join("outputs", "FDM_h_sensitivity")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 5000      # reduce for a quick local check, e.g. 1000-2000
PATIENCE = 3000
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

nx, ny = 51, 51
dx = 1.0 / (nx - 1)   # = 0.02, grid spacing used for the PDE/BC collocation grid

# Candidate step sizes as multiples of the collocation grid spacing dx
H_MULTIPLIERS = [0.125, 0.25, 0.5, 1.0, 2.0, 4.0]
H_VALUES = [m * dx for m in H_MULTIPLIERS]

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h_plate = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h_plate ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T


def analytical_w_nd(x, y):
    denom = np.pi ** 4 * D_nd * (1 / a_nd ** 2 + 1 / b_nd ** 2) ** 2
    return (q0_nd / denom) * np.sin(np.pi * x / a_nd) * np.sin(np.pi * y / b_nd)


def sinusoidal_load_nd(x, y):
    return q0_nd * torch.sin(np.pi * x / a_nd) * torch.sin(np.pi * y / b_nd)


w_anal_nd = analytical_w_nd(X_nd, Y_nd)
w_dim_anal = w_anal_nd * W
iy, ix = ny // 2, nx // 2
w_anal_center = w_dim_anal[iy, ix]

boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)
edge_x_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
edge_y_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
]))


# ============================================================
# MODEL  (identical to fdm_pinn_sinusoidal.py)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = []
        in_dim = 2
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.Tanh())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(0.01, dtype=DTYPE))

    def forward(self, x, y):
        h = torch.cat([x / a_nd, y / b_nd], dim=1)
        h = self.hidden(h)
        return self.scale * self.out(h)


def batched_stencil_eval(model, x, y, offsets):
    n = x.shape[0]
    xs = torch.cat([x + dx_ for dx_, dy_ in offsets], dim=0)
    ys = torch.cat([y + dy_ for dx_, dy_ in offsets], dim=0)
    w_all = model(xs, ys)
    return {off: w_all[i * n:(i + 1) * n] for i, off in enumerate(offsets)}


def fd_interior_derivatives(model, x, y, h):
    offsets = [
        (0, 0),
        (-2 * h, 0), (-h, 0), (h, 0), (2 * h, 0),
        (0, -2 * h), (0, -h), (0, h), (0, 2 * h),
        (h, h), (h, -h), (-h, h), (-h, -h),
    ]
    v = batched_stencil_eval(model, x, y, offsets)
    w_c = v[(0, 0)]
    w_xxxx = (v[(-2 * h, 0)] - 4 * v[(-h, 0)] + 6 * w_c - 4 * v[(h, 0)] + v[(2 * h, 0)]) / h ** 4
    w_yyyy = (v[(0, -2 * h)] - 4 * v[(0, -h)] + 6 * w_c - 4 * v[(0, h)] + v[(0, 2 * h)]) / h ** 4
    w_xxyy = (4 * w_c
              - 2 * (v[(h, 0)] + v[(-h, 0)] + v[(0, h)] + v[(0, -h)])
              + v[(h, h)] + v[(h, -h)] + v[(-h, h)] + v[(-h, -h)]) / h ** 4
    return w_c, w_xxxx, w_yyyy, w_xxyy


def fd_boundary_moments(model, x, y, h):
    offsets = [(0, 0), (-h, 0), (h, 0), (0, -h), (0, h)]
    v = batched_stencil_eval(model, x, y, offsets)
    w_c = v[(0, 0)]
    w_xx = (v[(-h, 0)] - 2 * w_c + v[(h, 0)]) / h ** 2
    w_yy = (v[(0, -h)] - 2 * w_c + v[(0, h)]) / h ** 2
    return w_c, w_xx, w_yy


def train_fdm_pinn(fd_step, run_tag):
    """Train one FDM-PINN with the given FD step size. Returns a results dict."""
    # reset seed so every h value starts from the same initialization
    np.random.seed(SEED)
    random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)
    bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
    int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)
    edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
    edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

    X_int = X_grid[int_idx]
    X_bc = X_grid[bc_idx]
    X_edge_x = X_grid[edge_x_idx]
    X_edge_y = X_grid[edge_y_idx]

    model = PlatePINN().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
    )

    def compute_loss():
        x_int, y_int = X_int[:, 0:1], X_int[:, 1:2]
        x_bc, y_bc = X_bc[:, 0:1], X_bc[:, 1:2]
        x_ex, y_ex = X_edge_x[:, 0:1], X_edge_x[:, 1:2]
        x_ey, y_ey = X_edge_y[:, 0:1], X_edge_y[:, 1:2]

        _, w_xxxx, w_yyyy, w_xxyy = fd_interior_derivatives(model, x_int, y_int, fd_step)
        q = sinusoidal_load_nd(x_int, y_int)
        pde_loss = torch.mean((w_xxxx + 2 * w_xxyy + w_yyyy - q) ** 2)

        bc_loss = torch.mean(model(x_bc, y_bc) ** 2)

        _, w_xx_ex, w_yy_ex = fd_boundary_moments(model, x_ex, y_ex, fd_step)
        Mx_edge = -(w_xx_ex + nu_nd * w_yy_ex)
        _, w_xx_ey, w_yy_ey = fd_boundary_moments(model, x_ey, y_ey, fd_step)
        My_edge = -(w_yy_ey + nu_nd * w_xx_ey)
        moment_loss = torch.mean(Mx_edge ** 2) + torch.mean(My_edge ** 2)

        total = pde_loss + bc_loss + moment_loss
        return total

    best_loss = float("inf")
    wait = 0
    epoch_times = []
    weights_path = os.path.join(OUT_DIR, f"weights_{run_tag}.pt")

    t0 = time.perf_counter()
    for epoch in range(MAX_EPOCHS):
        te0 = time.perf_counter()
        optimizer.zero_grad()
        loss = compute_loss()
        loss.backward()
        optimizer.step()
        scheduler.step()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        epoch_times.append(time.perf_counter() - te0)

        loss_val = loss.item()
        if loss_val < best_loss:
            best_loss = loss_val
            wait = 0
            torch.save(model.state_dict(), weights_path)
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    total_time = time.perf_counter() - t0

    model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    model.eval()

    with torch.no_grad():
        w_pred = model(X_grid[:, 0:1], X_grid[:, 1:2]).cpu().numpy().reshape(ny, nx)
    w_dim_pred = w_pred * W

    rel_L2 = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)
    center_pct_err = abs(w_dim_pred[iy, ix] - w_anal_center) / abs(w_anal_center) * 100

    os.remove(weights_path)  # sensitivity study only needs the metrics, not N_H weight files

    return {
        "fd_step_h": fd_step,
        "h_over_dx": fd_step / dx,
        "epochs_run": epoch + 1,
        "final_loss": best_loss,
        "mean_time_per_epoch_s": float(np.mean(epoch_times)),
        "total_training_time_s": total_time,
        "w_relative_L2_error": float(rel_L2),
        "center_pct_error": float(center_pct_err),
    }


# ============================================================
# RUN SWEEP
# ============================================================
results = []
for i, h in enumerate(H_VALUES):
    print(f"\n=== Training FDM-PINN with h = {h:.5f}  ({H_MULTIPLIERS[i]}x dx) "
          f"[{i+1}/{len(H_VALUES)}] ===")
    res = train_fdm_pinn(h, run_tag=f"h{i}")
    print(f"  -> rel L2 error = {res['w_relative_L2_error']:.4e}, "
          f"center % error = {res['center_pct_error']:.4f}, "
          f"mean s/epoch = {res['mean_time_per_epoch_s']:.4f}")
    results.append(res)

df = pd.DataFrame(results)
df.to_csv(os.path.join(OUT_DIR, "h_sensitivity_results.csv"), index=False)
print("\nSaved h_sensitivity_results.csv")
print(df.to_string(index=False))

# ============================================================
# RECOMMENDATION
# ============================================================
best_row = df.loc[df["w_relative_L2_error"].idxmin()]
recommendation = {
    "recommended_h": float(best_row["fd_step_h"]),
    "recommended_h_over_dx": float(best_row["h_over_dx"]),
    "rel_L2_error_at_recommended_h": float(best_row["w_relative_L2_error"]),
    "note": ("Error should trace a U-shape vs h: truncation error dominates for large h, "
             "floating-point cancellation error dominates for very small h. If the minimum "
             "falls at the edge of the tested range rather than in the interior, extend "
             "H_MULTIPLIERS further in that direction and re-run before finalizing h."),
}
with open(os.path.join(OUT_DIR, "h_recommendation.json"), "w") as f:
    json.dump(recommendation, f, indent=2)
print("\nRecommendation:", recommendation)

# ============================================================
# PLOTS
# ============================================================
plt.rcParams.update({"figure.dpi": 140})

fig, ax1 = plt.subplots(figsize=(8, 6))
ax1.plot(df["fd_step_h"], df["w_relative_L2_error"], "o-", color="tab:blue", label="Relative L2 error")
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("FD step size h (non-dimensional)")
ax1.set_ylabel("Relative L2 displacement error", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.axvline(x=dx, linestyle="--", color="gray", alpha=0.6, label="h = Δx (grid spacing)")
ax1.axvline(x=float(best_row["fd_step_h"]), linestyle=":", color="tab:red",
            label=f"Recommended h = {best_row['fd_step_h']:.4f}")
ax1.grid(True, which="both", alpha=0.3)
ax1.legend(loc="upper left")
plt.title("FDM-PINN: FD Step-Size Sensitivity (Sinusoidal Loading)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "h_sensitivity_error.png"))
plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df["fd_step_h"], df["mean_time_per_epoch_s"], "s-", color="tab:green")
plt.xscale("log")
plt.xlabel("FD step size h (non-dimensional)")
plt.ylabel("Mean training time per epoch (s)")
plt.title("FDM-PINN: Training Cost vs FD Step Size")
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "h_sensitivity_cost.png"))
plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")


=== Training FDM-PINN with h = 0.00250  (0.125x dx) [1/6] ===


C:\Users\vivek\AppData\Local\Temp\ipykernel_6604\179173227.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location=

  -> rel L2 error = 3.4551e-01, center % error = 24.8786, mean s/epoch = 0.0524

=== Training FDM-PINN with h = 0.00500  (0.25x dx) [2/6] ===
  -> rel L2 error = 3.4596e-01, center % error = 24.8582, mean s/epoch = 0.0469

=== Training FDM-PINN with h = 0.01000  (0.5x dx) [3/6] ===
  -> rel L2 error = 3.4587e-01, center % error = 24.8953, mean s/epoch = 0.0282

=== Training FDM-PINN with h = 0.02000  (1.0x dx) [4/6] ===
  -> rel L2 error = 3.4602e-01, center % error = 24.9005, mean s/epoch = 0.0280

=== Training FDM-PINN with h = 0.04000  (2.0x dx) [5/6] ===
  -> rel L2 error = 3.4712e-01, center % error = 25.0213, mean s/epoch = 0.0282

=== Training FDM-PINN with h = 0.08000  (4.0x dx) [6/6] ===
  -> rel L2 error = 3.5514e-01, center % error = 25.9405, mean s/epoch = 0.0280

Saved h_sensitivity_results.csv
 fd_step_h  h_over_dx  epochs_run  final_loss  mean_time_per_epoch_s  total_training_time_s  w_relative_L2_error  center_pct_error
    0.0025      0.125        5000    0.000003     